In [1]:
#fish bin
from sklearn import preprocessing
import pandas as pd
import joblib
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from rdkit import Chem
from rdkit.Chem import MACCSkeys
Xtrain= pd.read_excel(r"Xtrain索引fish.xlsx")
Xtest= pd.read_excel(r"Xtest索引fish.xlsx")
Ytrain= pd.read_excel(r"Ytrain索引fish.xlsx")
Ytest= pd.read_excel(r"Ytest索引fish.xlsx")
Xtrain=Xtrain.iloc[:,1:]
Xtest=Xtest.iloc[:,1:]
Ytrain=Ytrain.iloc[:,1:]
Ytest=Ytest.iloc[:,1:]
xStandardScaler_Xtrain=Xtrain.iloc[:,0:35]
scaler=preprocessing.StandardScaler().fit(xStandardScaler_Xtrain)
xStandard_Xtrain=scaler.transform(xStandardScaler_Xtrain)
# 假设你有一个名为 xStandardScaler_Xtrain 的 DataFrame
# 创建一个 StandardScaler 对象并进行拟合和转换
scaler = preprocessing.StandardScaler().fit(xStandardScaler_Xtrain)
xStandard_Xtrain = scaler.transform(xStandardScaler_Xtrain)
# 将标准化后的数据重新放入一个新的 DataFrame，保留列名称
xStandard_Xtrain_df = pd.DataFrame(xStandard_Xtrain, columns=xStandardScaler_Xtrain.columns)
bin2 =pd.read_excel(r"D:/cpython/python/to pre fishbin/没有smote.xlsx")
bin_st = bin2.iloc[:,3:38]
bin_sp = bin2.iloc[:,38:]
bin1 = pd.DataFrame(scaler.transform(bin_st), columns=bin_st.columns)
bin1=pd.DataFrame(bin1)
new_data1 = pd.concat([bin1, bin_sp], axis=1)
Xtrainmodel = pd.read_excel("Xtrain标准完fish.xlsx")
Xtestmodel=pd.read_excel("Xtest标准完fish.xlsx")
# 加载保存的模型
model_filename = 'GB_cls_to pre fishbin final_122.pkl'  # 模型文件的名称，确保它在当前工作目录或提供完整的路径
loaded_model = joblib.load(model_filename)
# 使用加载的模型进行预测
predictions = loaded_model.predict(new_data1)
predictions =pd.DataFrame(predictions)




In [3]:
predictions.to_excel("D:/cpython/python/to pre fishbin/pred bin.xlsx")

In [ ]:
##########################读取建模数据
data = pd.read_excel(r"D:/cpython/python/to pre fishbin/没有smote.xlsx")
# 为SMILES列创建RDKit分子对象
data['Molecule'] = data['smiles'].apply(Chem.MolFromSmiles)
# 计算MACCS指纹
data['MACCS'] = data['Molecule'].apply(MACCSkeys.GenMACCSKeys)
# 将MACCS指纹拆分为167列
maccs_columns = [f'MACCS_{i+1}' for i in range(167)]
data[maccs_columns] = data['MACCS'].apply(lambda x: pd.Series(list(x), dtype=int))
features_lab = data[['Clhuman', 'LogD55_pred',	'LogD74_pred']]
frature = pd.concat([features_lab, data.iloc[:,38:61],data.iloc[:,63:]],axis=1)#data.iloc[:,40:63],
# 标准化数据
scaler = StandardScaler()
scaled_data = scaler.fit_transform(features_lab)
scaled_data=pd.DataFrame(scaled_data)
scaled_data1= pd.concat([scaled_data, data.iloc[:,38:61],data.iloc[:,63:]],axis=1)#data.iloc[:,40:63],
# 执行PCA降维
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(scaled_data)
df_reduced = pd.DataFrame(data=reduced_data, columns=['PC1', 'PC2'])
df=df_reduced
from scipy.stats import gaussian_kde
# 假设有两列数据，分别是 x 和 y
x_data = df['PC1']
y_data = df['PC2']
data = np.vstack([x_data, y_data])
# 计算核密度估计
kde = gaussian_kde(data)
# 对于每个数据点，计算其核密度值
density_values = kde(data)
# 计算最小值和最大值
min_density = np.min(density_values)
max_density = np.max(density_values)
# 最小-最大标准化
normalized_density = (density_values - min_density) / (max_density - min_density) * 100
# 创建DataFrame
df_output = pd.DataFrame({
    'Standardized Density': normalized_density,
    'AD': (normalized_density > 10).astype(int)  # 根据大于0.01的条件填充1，小于等于0.01填充0
})
contat = pd.concat([predictions, df_output], axis=1)
contat2 = pd.concat([contat.iloc[:, [0]], contat.iloc[:, [2]]], axis=1)

# 修改第一列的列名为"Bin3"，第二列保持原列名或者也可以给它指定一个新名字
contat2.columns = ['Bin3 ', contat2.columns[1]]  # 如果需要修改第二列的名字，可以在这里指定
d = pd.read_excel("IVIVE的体内数据2.xlsx",sheet_name="logD")
contatt = pd.concat([contat2, d], axis=1)
contatt['AD_final'] = (contatt['AD'] & contatt['AD_LogD']& contatt['AD_Clint']).astype(int)
contatt_final =contatt[['Bin3 ','AD_final']]
contatt_final.rename(columns={'AD_final': 'AD'}, inplace=True)
contatt_final.to_excel("fishbin AD.xlsx", index=False)

In [6]:
#定量 one-hot处理没有加
Xtrain=pd.read_excel("Xtrain索引eu.xlsx")
Xtest= pd.read_excel("Xtest索引eu.xlsx")
Ytrain= pd.read_excel("Ytrain索引eu.xlsx")
Ytest= pd.read_excel("Ytest索引eu.xlsx")
xStandardScaler_Xtrain=Xtrain.iloc[:,4:]
xStandardScaler_Xtrain
column_names = xStandardScaler_Xtrain.columns
# 标准化数据
scaler = preprocessing.StandardScaler().fit(xStandardScaler_Xtrain)
xStandard_Xtrain = scaler.transform(xStandardScaler_Xtrain)
# 创建带有列名称的DataFrame
df_standardized = pd.DataFrame(xStandard_Xtrain, columns=column_names)
a=pd.read_excel("IVIVE的体内数据2.xlsx",sheet_name='fish cl')

a_st=a.iloc[:,2:-24]
a_st=pd.DataFrame(a_st)
a_sp=a.iloc[:,-24:]

Xt = pd.DataFrame(scaler.transform(a_st), columns=a_st.columns)
input_fish=pd.concat([Xt, a_sp], axis=1)

Xtrainmodel=pd.read_excel("Xtrain所有X标准化eu.xlsx")
Xtestmodel=pd.read_excel("Xtest所有X标准化eu.xlsx")
Xtrainmodel=Xtrainmodel.iloc[:,1:80]
Xtrainmodel=pd.DataFrame(Xtrainmodel)
Xtestmodel=Xtestmodel.iloc[:,1:80]
Xtestmodel=pd.DataFrame(Xtestmodel)
Ytrain=Ytrain.iloc[:,1]
Ytrain=pd.DataFrame(Ytrain)
Ytest=Ytest.iloc[:,1]
Ytest=pd.DataFrame(Ytest)
# 加载保存的模型
model_filename = 'eu de fish model_122.pkl'  # 模型文件的名称，确保它在当前工作目录或提供完整的路径
loaded_model = joblib.load(model_filename)
#input_fish=pd.read_excel("IVIVE的体内数据2.xlsx",sheet_name='fish cl标准完')
#input_fish=input_fish.iloc[:, 1:]
# 使用加载的模型进行预测
predictions = loaded_model.predict(input_fish)
predictions =pd.DataFrame(predictions)

#应用域
V1y_pred=pd.read_excel("VOTE测试集预测值.xlsx")
Ytest=pd.read_excel("VOTE测试集实验值.xlsx")
Ytrain=pd.read_excel("VOTE训练集实验值.xlsx")
V1y0_pred=pd.read_excel("VOTE训练集预测值.xlsx")
V1y_pred=pd.DataFrame(V1y_pred)
V1y0_pred=pd.DataFrame(V1y0_pred)
V1y0_pred = loaded_model.predict(Xtrainmodel)
V1y0_pred=pd.DataFrame(V1y0_pred)
residual_train= Ytrain - V1y0_pred
residual_train= Ytrain - V1y0_pred
residual_test= Ytest - V1y_pred
s_residual_train = ((residual_train) - residual_train.mean()) / np.std(residual_train, ddof = 1)
s_residual_test = ((residual_test) - residual_test.mean()) / np.std(residual_test, ddof = 1)
tXmodel = np.transpose(Xtrainmodel)
xtx = np.dot(tXmodel,Xtrainmodel)
invxtx = np.linalg.inv(xtx)
hat_matrix0 = np.dot(Xtrainmodel,invxtx)
hat_matrix = np.dot(hat_matrix0,tXmodel)
hi = np.diagonal(hat_matrix)
hi.max()
p = 59
n = len(Xtrainmodel) #+nrow(X_test)#training compounds
h_star = (3*p)/n

tXmodeltest = np.transpose(Xtestmodel)
hat_matrix0test = np.dot(Xtestmodel,invxtx)
hat_matrixtest = np.dot(hat_matrix0test,tXmodeltest)
hitest = np.diagonal(hat_matrixtest)
xsettest=input_fish
tXexampletest = np.transpose(xsettest)
hat_matrix0exampletest = np.dot(xsettest,invxtx)
hat_matrixexampletest = np.dot(hat_matrix0exampletest,tXexampletest)
hiexampletest = np.diagonal(hat_matrixexampletest)
pd.DataFrame(predictions).to_excel('fish cl pred.xlsx')

existing_excel = pd.read_excel('fish cl pred.xlsx')
hiexampletest_df = pd.DataFrame(hiexampletest)
existing_excel.insert(2, 'AD', hiexampletest_df)
existing_excel.to_excel('fish cl pred.xlsx', index=False)
df = pd.read_excel('fish cl pred.xlsx')

for index, value in df['AD'].items():
    if value > 0.804:
        df.at[index, 'AD'] = 0
    else:
        df.at[index, 'AD'] = 1
df = df.drop(columns=df.columns[0])
df = df.rename(columns={df.columns[0]: 'Cl'})
df['Cl'] = (10 ** df['Cl']) * 24 * 60 * 510 / 1000
df['Cl'] = df['Cl'].round(2)
df['AD2'] = contatt_final['AD']
df['AD'] = df['AD'].astype(int)
df['AD2'] = df['AD2'].astype(int)

# Calculate AD_final based on conditions
df['AD_final'] = (df['AD'] & df['AD2']).astype(int)
df = df.drop(columns=df.columns[1])
df = df.drop(columns=df.columns[1])
df.to_excel('fish cl pred.xlsx', index=False)

C:\Users\82497\.conda\envs\torch_skl122\lib\site-packages\numpy\core\fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
